<a href="https://colab.research.google.com/github/the-name-is-madhu/BIS_LAB/blob/main/1BM24CS154_Disaster_Relief_Material_Distribution___Parrallel_Cellular_Algorithm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:

import random
from concurrent.futures import ThreadPoolExecutor

# ---------------------------------
# DISASTER RELIEF INPUT DATA
# ---------------------------------

regions = ["R1", "R2", "R3", "R4", "R5"]

materials = ["Food", "Water", "Medicine"]

# Required materials for each region
# Rows = regions, Columns = materials

demand = [
    [500, 1000, 100],
    [300, 600, 60],
    [700, 1400, 140],
    [400, 800, 80],
    [600, 1200, 120]
]

# Available warehouse inventory
inventory = [1800, 3600, 360]

# Region priority weights
priority = [2, 3, 5, 2, 4]

# Transportation distances (km)
distance = [10, 20, 35, 15, 25]

# Algorithm parameters
NUM_CELLS = 30
MAX_ITER = 100

ALPHA = 10
BETA = 0.05


# ---------------------------------
# INITIALIZE CELL
# ---------------------------------

def initialize_cell():

    allocation = []

    for i in range(len(regions)):

        row = []

        for j in range(len(materials)):

            value = random.randint(
                0, demand[i][j]
            )

            row.append(value)

        allocation.append(row)

    return repair(allocation)


# ---------------------------------
# REPAIR INVENTORY CONSTRAINTS
# ---------------------------------

def repair(allocation):

    allocation = [
        row.copy() for row in allocation
    ]

    for j in range(len(materials)):

        total = sum(
            allocation[i][j]
            for i in range(len(regions))
        )

        while total > inventory[j]:

            i = random.randrange(len(regions))

            if allocation[i][j] > 0:
                allocation[i][j] -= 1
                total -= 1

    return allocation


# ---------------------------------
# FITNESS EVALUATION
# ---------------------------------

def fitness(allocation):

    unmet_demand = 0
    transport_cost = 0

    for i in range(len(regions)):

        for j in range(len(materials)):

            required = demand[i][j]
            supplied = allocation[i][j]

            shortage = max(
                0, required - supplied
            )

            unmet_demand += (
                shortage * priority[i]
            )

            transport_cost += (
                supplied * distance[i]
            )

    return (
        ALPHA * unmet_demand
        + BETA * transport_cost
    )


# ---------------------------------
# NEIGHBOURHOOD UPDATE
# ---------------------------------

def update_cell(index, old_cells):

    neighbours = [
        old_cells[(index - 1) % NUM_CELLS],
        old_cells[index],
        old_cells[(index + 1) % NUM_CELLS]
    ]

    best_neighbour = min(
        neighbours,
        key=fitness
    )

    current = old_cells[index]

    new_solution = []

    for i in range(len(regions)):

        row = []

        for j in range(len(materials)):

            # Learn from neighbouring solution
            if random.random() < 0.6:

                value = best_neighbour[i][j]

            else:

                value = current[i][j]

            # Random exploration
            if random.random() < 0.2:

                value += random.randint(-50, 50)

            value = max(
                0,
                min(value, demand[i][j])
            )

            row.append(value)

        new_solution.append(row)

    return repair(new_solution)


# ---------------------------------
# PARALLEL CELLULAR ALGORITHM
# ---------------------------------

def parallel_cellular_algorithm():

    cells = [
        initialize_cell()
        for _ in range(NUM_CELLS)
    ]

    best_solution = None
    best_score = float("inf")

    with ThreadPoolExecutor() as executor:

        for iteration in range(MAX_ITER):

            # Evaluate all cells in parallel
            scores = list(
                executor.map(fitness, cells)
            )

            # Find global best solution
            for i, score in enumerate(scores):

                if score < best_score:

                    best_score = score

                    best_solution = [
                        row.copy()
                        for row in cells[i]
                    ]

            # Snapshot cellular states
            old_cells = [
                [row.copy() for row in cell]
                for cell in cells
            ]

            # Update all cells in parallel
            cells = list(
                executor.map(
                    lambda i: update_cell(
                        i, old_cells
                    ),
                    range(NUM_CELLS)
                )
            )

    return best_solution, best_score


# ---------------------------------
# DISPLAY OPTIMIZED DISTRIBUTION
# ---------------------------------

solution, score = parallel_cellular_algorithm()

print("\nOPTIMIZED DISASTER RELIEF PLAN")

for i, region in enumerate(regions):

    print("\nRegion:", region)

    for j, material in enumerate(materials):

        print(
            material, ":", solution[i][j]
        )

print("\nMinimum Fitness:", score)

print("\nTOTAL RESOURCE DISTRIBUTION")

for j, material in enumerate(materials):

    total = sum(
        solution[i][j]
        for i in range(len(regions))
    )

    print(material, ":", total)

print("\nREMAINING INVENTORY")

for j, material in enumerate(materials):

    total = sum(
        solution[i][j]
        for i in range(len(regions))
    )

    print(
        material, ":",
        inventory[j] - total
    )



OPTIMIZED DISASTER RELIEF PLAN

Region: R1
Food : 268
Water : 371
Medicine : 61

Region: R2
Food : 290
Water : 542
Medicine : 1

Region: R3
Food : 693
Water : 1398
Medicine : 140

Region: R4
Food : 0
Water : 97
Medicine : 63

Region: R5
Food : 549
Water : 1192
Medicine : 81

Minimum Fitness: 56064.75

TOTAL RESOURCE DISTRIBUTION
Food : 1800
Water : 3600
Medicine : 346

REMAINING INVENTORY
Food : 0
Water : 0
Medicine : 14
